<a href="https://colab.research.google.com/github/westcastleone/FairTradingSTG/blob/main/Daily_Aftrer_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# 라이브러리 설치
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 1.3 MB/s eta 0:00:00


In [ ]:
from abc import ABC, abstractmethod

# 공통 클래스 정의
class StockDataLoader(ABC):
    @abstractmethod
    def stock_fetch_data(self, interval):
        pass

from datetime import datetime
today_date = datetime.now().date()

In [ ]:
import requests
import json
import pandas as pd

# 한국투자증권 Loader 클래스 정의
class KisLoader(StockDataLoader):
  def __init__(self, api_key, api_secret):
    self.api_key = api_key
    self.api_secret = api_secret
    self.token = self.get_access_token()

  def get_access_token(self):
    url = "https://openapi.koreainvestment.com:9443/oauth2/tokenP" # 실전투자 기준

    headers = {"content-type":"application/json"}

    body = {
        "grant_type": "client_credentials",
        "appkey": self.api_key,
        "appsecret": self.api_secret
    }

    res = requests.post(url, headers=headers, data=json.dumps(body))

    if res.status_code == 200:
        return res.json()["access_token"]
    else:
        print("토큰 발급 실패:", res.text)
        return None

  def stock_fetch_data(self, interval):
    url = "https://openapi.koreainvestment.com:9443/uapi/domestic-stock/v1/quotations/inquire-time-indexchartprice" # 실전투자 기준

    headers = {
        "content-type": "application/json; charset=utf-8",
        "authorization": f"Bearer {self.token}",
        "appkey": self.api_key,
        "appsecret": self.api_secret,
        "tr_id": "FHPUP02110200",
        "custtype": "P",
    }

    # 쿼리 파라미터 설정 (코스피, 코스닥 1분봉)
    kospi_params = {
      "FID_COND_MRKT_DIV_CODE": "U",
      "FID_INPUT_ISCD": "0001",
      "FID_INPUT_HOUR_1": interval
    }

    kosdaq_params = {
      "FID_COND_MRKT_DIV_CODE": "U",
      "FID_INPUT_ISCD": "1001",
      "FID_INPUT_HOUR_1": interval
    }

    # API 호출
    kospi_res = requests.get(url, headers=headers, params=kospi_params)
    kospi_data = kospi_res.json()

    kosdaq_res = requests.get(url, headers=headers, params=kosdaq_params)
    kosdaq_data = kosdaq_res.json()

    # 데이터 전처리
    df_kospi = pd.DataFrame(kospi_data['output'])[['bsop_hour','bstp_nmix_prpr']]
    df_kospi.columns = ['bsop_hour', 'kospi'] # 컬럼명 명확하게 변경

    df_kosdaq = pd.DataFrame(kosdaq_data['output'])[['bsop_hour','bstp_nmix_prpr']]
    df_kosdaq.columns = ['bsop_hour', 'kosdaq'] # 컬럼명 명확하게 변경

    # bsop_hour 기준으로 innner join
    merged_df = pd.merge(df_kospi, df_kosdaq, on='bsop_hour')

    # 시간이 150000 부터 153000 까지만 가져오도록 조정
    filtered_df = merged_df[
      (merged_df['bsop_hour'].astype(int) >= 150000) &
      (merged_df['bsop_hour'].astype(int) <= 153000)
    ]


    filtered_df = filtered_df.rename(columns={'bsop_hour': 'dateTime'})

    # yf 형식으로 바꾸기
    today = datetime.now().strftime('%Y-%m-%d ')
    filtered_df['dateTime'] = pd.to_datetime(today + filtered_df['dateTime'], format='%Y-%m-%d %H%M%S')

    filtered_df.set_index('dateTime', inplace=True)

    refined_data = [
        (index, float(row['kospi']), float(row['kosdaq']))
        for index, row in filtered_df.iterrows()
    ]

    # 최종으로 DataFrame 형식으로 변환
    kis_df = pd.DataFrame(refined_data, columns=['dateTime', 'kospi', 'kosdaq'])

    return kis_df


In [ ]:
import time as t
import random
import yfinance as yf
from datetime import time

# 야후파이넨셜 클래스 정의
class YahooLoader(StockDataLoader):
  def get_yahoo_data(self, ticker_symbol, period, interval):
    # time.sleep(random.uniform(1, 3))
    df = yf.download(ticker_symbol, period=period, interval=interval)

    # yfinance의 멀티인덱스 컬럼을 단일층으로 평탄화
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()

    return df

  def stock_fetch_data(self, interval):
    # 데이터 가져오기
    df_kospi = self.get_yahoo_data('^KS11', '1d', interval)
    t.sleep(random.uniform(1, 3))
    df_kosdaq = self.get_yahoo_data('^KQ11', '1d', interval)

    # dataTime과 Close 말고는 슬라이싱
    df_kospi = df_kospi[['Datetime', 'Close']].rename(columns={'Datetime': 'dateTime', 'Close': 'kospi'})
    df_kosdaq = df_kosdaq[['Datetime', 'Close']].rename(columns={'Datetime': 'dateTime', 'Close': 'kosdaq'})

    # 데이터 전처리
    merged_df = pd.merge(df_kospi, df_kosdaq, on='dateTime')

    # yfinance 데이터 한국 시간으로 변환
    merged_df['dateTime'] = pd.to_datetime(merged_df['dateTime']).dt.tz_convert('Asia/Seoul')

    # 09:00~15:00 데이터만 가져오기
    filtered_df = merged_df[
        (merged_df['dateTime'].dt.time >= time(9, 0, 0)) &
        (merged_df['dateTime'].dt.time < time(15, 0, 0))
    ].copy()

    # DataTime 타임존 정보 제거
    filtered_df['dateTime'] = pd.to_datetime(filtered_df['dateTime']).dt.tz_localize(None)

    return filtered_df

  def get_macro_data(self, interval):
    # 데이터 가져오기
    df_USDKRW = self.get_yahoo_data('USDKRW=X', '2d', interval)
    df_DXY = self.get_yahoo_data('DX-Y.NYB', '2d', interval)

    # dataTime과 Close 말고는 슬라이싱
    df_USDKRW = df_USDKRW[['Datetime', 'Close']].rename(columns={'Datetime': 'dateTime', 'Close': 'USD/KRW'})
    df_DXY = df_DXY[['Datetime', 'Close']].rename(columns={'Datetime': 'dateTime', 'Close': 'DXY'})

    # 데이터 전처리
    merged_df = pd.merge(df_USDKRW, df_DXY, on='dateTime')

    # DataTime 한국 시간으로 변경 및 타임존 정보 제거
    merged_df['dateTime'] = pd.to_datetime(merged_df['dateTime']).dt.tz_convert('Asia/Seoul').dt.tz_localize(None)

    print(merged_df)

    # 날짜가 오늘이고, 시간이 09:00:00 이상 15:30:00 이하인 데이터만 마스킹
    mask = (merged_df['dateTime'].dt.date == today_date) & \
           (merged_df['dateTime'].dt.time <= time(15, 30, 0))

    filtered_macro = merged_df.loc[mask].copy()
    return filtered_macro

  def get_perormance_summary(self):
    # 코스피, 코스닥 1년 기준 데이터 가져오기
    df_kospi = self.get_yahoo_data('^KS11', period='1y', interval='1d')
    df_kosdaq = self.get_yahoo_data('^KQ11', period='1y', interval='1d')

    # dataTime과 Open Close 말고는 슬라이싱
    df_kospi = df_kospi[['Date', 'Open', 'Close']]
    df_kosdaq = df_kosdaq[['Date', 'Open', 'Close']]

    # 코스피, 코스닥 현재가
    current_kospi_price = df_kospi['Close'].iloc[-1]
    current_kosdaq_price = df_kosdaq['Close'].iloc[-1]

    # 코스피, 코스닥 Current, 1D, 1W, 1M, YTD 계산
    kospi_perf = {
            "Current": current_kospi_price,
            "1D": (current_kospi_price / df_kospi['Close'].iloc[-2] - 1),
            "1W": (current_kospi_price / df_kospi['Close'].iloc[-6] - 1),
            "1M": (current_kospi_price / df_kospi['Close'].iloc[-22] - 1),
            "YTD": (current_kospi_price / df_kospi[df_kospi['Date'].dt.year == datetime.now().year]['Open'].iloc[0] - 1)
    }
    kosdaq_perf = {
            "Current": current_kosdaq_price,
            "1D": (current_kosdaq_price / df_kosdaq['Close'].iloc[-2] - 1),
            "1W": (current_kosdaq_price / df_kosdaq['Close'].iloc[-6] - 1),
            "1M": (current_kosdaq_price / df_kosdaq['Close'].iloc[-22] - 1),
            "YTD": (current_kosdaq_price / df_kosdaq[df_kosdaq['Date'].dt.year == datetime.now().year]['Open'].iloc[0] - 1)
    }
    return {"KOSPI" : kospi_perf, "KOSDAQ" : kosdaq_perf}

In [ ]:
# API Key Setting (API Key 외부 유출 금지!!)
api_key = ""
api_secret = ""

yahoo = YahooLoader()
kis = KisLoader(api_key, api_secret)

# Loader 클래스를 통해 일일 Stock 데이터 가져오기
stock_yahoo_df = yahoo.stock_fetch_data("1m")
stock_kis_df = kis.stock_fetch_data("60")

# Stock 데이터 연결 및 시간 정렬
stock_df = pd.concat([stock_yahoo_df, stock_kis_df], ignore_index=True)
stock_df = stock_df.sort_values(by='dateTime').reset_index(drop=True)

## 엑셀이 날짜로 오해하지 못하게 문자열 컬럼 생성
stock_df['time_label'] = stock_df['dateTime'].dt.strftime('%H:%M')

## 데이터 타입 변환
stock_df['kospi'] = pd.to_numeric(stock_df['kospi'], errors='coerce')
stock_df['kosdaq'] = pd.to_numeric(stock_df['kosdaq'], errors='coerce')

## 환율 데이터 가져오기
macro_df = yahoo.get_macro_data('1m')

## 엑셀이 날짜로 오해하지 못하게 문자열 컬럼 생성
macro_df['time_label'] = macro_df['dateTime'].dt.strftime('%H:%M')

## 데이터 타입 변환
macro_df['USD/KRW'] = pd.to_numeric(macro_df['USD/KRW'], errors='coerce')
macro_df['DXY'] = pd.to_numeric(macro_df['DXY'], errors='coerce')

## Summary 정보 가져오기
perormance_summary = yahoo.get_perormance_summary()

토큰 발급 실패: {"error_code":"EGW00102","error_description":"AppKey는 필수입니다."}


/tmp/ipykernel_959/921339153.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker_symbol, period=period, interval=interval)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_959/921339153.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker_symbol, period=period, interval=interval)
[*********************100%***********************]  1 of 1 completed


KeyError: 'output'

In [ ]:
# perormance_summary를 dict 형태로 변환
performance_df = pd.DataFrame.from_dict(perormance_summary, orient='index')

In [ ]:
import pytz # 시간대 처리를 위한 라이브러리
from google.colab import files # 파일 다운로드를 위한 라이브러리

# 엑셀
file_name = f'Morning_Report_{today_date.strftime("%Y%m%d")}.xlsx'
writer = pd.ExcelWriter(file_name, engine='xlsxwriter')

# 시트 생성
workbook = writer.book
worksheet = workbook.add_worksheet('Dashboard')

# 포맷 정의 (상단 제목용, 헤더용, 숫자용)
title_format = workbook.add_format({'bold': True, 'font_size': 14})
header_format = workbook.add_format({
    'bold': True,
    'align': 'center',
    'valign': 'vcenter',
    'bg_color': '#D9E1F2',  # 연한 파란색 배경
    'border': 1
})
base_format = workbook.add_format({'align': 'center', 'border': 1}) # 기본 테두리

## 서식 적용($, #)
num_fmt = workbook.add_format({'num_format': '#,##0.00', 'align': 'center', 'border': 1})
percent_fmt = workbook.add_format({'num_format': '0.00%', 'align': 'center', 'border': 1})

pos_format = workbook.add_format({'font_color': 'red', 'num_format': '+0.00%', 'border': 1, 'align': 'center'})
neg_format = workbook.add_format({'font_color': 'blue', 'num_format': '0.00%', 'border': 1, 'align': 'center'})

# 제목 지정
worksheet.write('B2', '■ Market Summary', title_format)

# Sumaary 데이터
headers = ['Index', 'Current', '1D', '1W', '1M', 'YTD']
for col_num, header in enumerate(headers):
    worksheet.write(3, col_num + 1, header, header_format)

for row_num, (idx_name, row_data) in enumerate(performance_df.iterrows()):
    curr_row = row_num + 4  # 시작 행 위치 (B5 셀부터)

    ## Index (B열): KOSPI, KOSDAQ
    worksheet.write(curr_row, 1, idx_name, base_format)

    ## Current (C열): 현재가 (숫자 포맷 적용)
    worksheet.write(curr_row, 2, row_data['Current'], num_fmt)

    ## 1D, 1W, 1M, YTD (D~G열): 수익률 (퍼센트 포맷 적용)
    worksheet.write(curr_row, 3, row_data['1D'], percent_fmt)
    worksheet.write(curr_row, 4, row_data['1W'], percent_fmt)
    worksheet.write(curr_row, 5, row_data['1M'], percent_fmt)
    worksheet.write(curr_row, 6, row_data['YTD'], percent_fmt)



worksheet.set_column('B:B', 15)
worksheet.set_column('C:C', 15)
worksheet.set_column('D:G', 12)

## 조건부 서식 지정
worksheet.conditional_format('D5:G6', {
    'type':     'cell',
    'criteria': '>',
    'value':    0,
    'format':   workbook.add_format({'font_color': 'red', 'num_format': '+0.00%'}),
})

worksheet.conditional_format('D5:G6', {
    'type':     'cell',
    'criteria': '<',
    'value':    0,
    'format':   workbook.add_format({'font_color': 'blue', 'num_format': '0.00%'}),
})

In [ ]:
# Stock 그래프 그리기
stock_df.to_excel(writer, sheet_name='Data_Storage', startrow=1, startcol=0, index=False)
max_row = len(stock_df) + 1

# 차트 객체 생성 및 투명도 설정
kos_chart = workbook.add_chart({'type': 'line'})

kos_chart.set_chartarea({'fill': {'none': True}, 'border': {'none': True}})
kos_chart.set_plotarea({'fill': {'none': True}, 'border': {'none': True}})

# 시리즈 추가 (categories는 D열(3번 인덱스)인 time_label을 참조)
kos_chart.add_series({
    'name':       'KOSPI',
    'categories': ['Data_Storage', 2, 3, max_row, 3], # D열: HH:MM 문자열
    'values':     ['Data_Storage', 2, 1, max_row, 1], # B열: kospi
    'line':       {'color': '#C00000', 'width': 1.5},
})

kos_chart.add_series({
    'name':       'KOSDAQ',
    'categories': ['Data_Storage', 2, 3, max_row, 3], # D열: HH:MM 문자열
    'values':     ['Data_Storage', 2, 2, max_row, 2], # C열: kosdaq
    'y2_axis':    True,
    'line':       {'color': '#002060', 'width': 1.5},
})

kos_chart.set_y_axis({
    'major_gridlines': {'visible': False}, # 가로선 제거
    'line': {'none': True}                 # Y축 선도 제거 (선택사항)
})

kos_chart.set_y2_axis({
    'major_gridlines': {'visible': False}, # 보조축 가로선도 제거
    'line': {'none': True}
})

# X축 시간 표시 및 눈금 제거
kos_chart.set_x_axis({
    'interval_unit': 90,          # 30분 단위 라벨
    'major_tick_mark': 'none',    # 눈금선 제거
    'label_position': 'low',
    'line': {'none': True},       # X축 바닥 선 제거
})

kos_chart.set_size({'width': 750, 'height': 450})
kos_chart.set_legend({'position': 'bottom'})

# Dashboard 시트에 삽입
worksheet.insert_chart('B8', kos_chart)

In [ ]:
# Macro 그래프 그리기
macro_df.to_excel(writer, sheet_name='Data_Storage', startrow=1, startcol=5, index=False)
max_row = len(macro_df) + 1

# 차트 객체 생성 및 투명도 설정
macro_chart = workbook.add_chart({'type': 'line'})

macro_chart.set_chartarea({'fill': {'none': True}, 'border': {'none': True}})
macro_chart.set_plotarea({'fill': {'none': True}, 'border': {'none': True}})

# 시리즈 추가 (categories는 I열(3번 인덱스)인 time_label을 참조)
macro_chart.add_series({
    'name':       'USD/KRW',
    'categories': ['Data_Storage', 2, 8, max_row, 8], # I열: HH:MM 문자열
    'values':     ['Data_Storage', 2, 6, max_row, 6], # G열: USD/KRW
    'line':       {'color': '#5F9EA0', 'width': 1.5},
})

macro_chart.add_series({
    'name':       'DXY',
    'categories': ['Data_Storage', 2, 8, max_row, 8], # I열: HH:MM 문자열
    'values':     ['Data_Storage', 2, 7, max_row, 7], # H열: DXY
    'y2_axis':    True,
    'line':       {'color': '#FF7F50', 'width': 1.5},
})

macro_chart.set_y_axis({
    'major_gridlines': {'visible': False}, # 가로선 제거
    'line': {'none': True}                 # Y축 선도 제거 (선택사항)
})

macro_chart.set_y2_axis({
    'major_gridlines': {'visible': False}, # 보조축 가로선도 제거
    'line': {'none': True}
})

# X축 시간 표시 및 눈금 제거
macro_chart.set_x_axis({
    'interval_unit': 90,          # 30분 단위 라벨
    'major_tick_mark': 'none',    # 눈금선 제거
    'label_position': 'low',
    'line': {'none': True},       # X축 바닥 선 제거
})

macro_chart.set_size({'width': 750, 'height': 450})
macro_chart.set_legend({'position': 'bottom'})

# Dashboard 시트에 삽입
worksheet.insert_chart('B31', macro_chart)

In [ ]:
# 헤더 텍스트 추가
dataStorage_worksheet = writer.sheets['Data_Storage']

header_format = workbook.add_format({'bold': True, 'font_size': 12})
dataStorage_worksheet.write('A1', 'Stock_Intra', header_format)
dataStorage_worksheet.write('F1', 'Macro_Intra', header_format)

In [ ]:
writer.close()
print(f"✅ 엑셀 파일 생성 완료: {file_name}")

# 6. 파일 다운로드 (코랩 전용 기능)
files.download(file_name)